# 01 — Mountain NER Dataset Creation

This notebook explains the construction and validation of the dataset used to
fine-tune a mountain-name NER model. The dataset uses BIO labels.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../data/processed")


## Dataset design

The baseline contains:
- synthetic sentences with one mountain;
- synthetic sentences with two mountains;
- ordinary negative examples;
- hard negatives containing ambiguous terms and names.

The synthetic origin is a limitation and is not hidden.


In [ ]:
with open(DATA_DIR / "dataset_stats.json", encoding="utf-8") as f:
    stats = json.load(f)

pd.DataFrame(stats).T


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
train.sample(10, random_state=42)[["text", "tokens", "ner_tags", "source_type"]]


## Quality checks

Each example is checked for:
1. equal token and label lengths;
2. valid BIO labels;
3. entity offsets matching the source text;
4. no annotations in negative examples.


In [ ]:
VALID_LABELS = {"O", "B-MOUNTAIN", "I-MOUNTAIN"}

for split in ["train", "validation", "test"]:
    path = DATA_DIR / f"{split}.jsonl"
    with path.open(encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            assert len(row["tokens"]) == len(row["ner_tags"])
            assert set(row["ner_tags"]).issubset(VALID_LABELS)
            for entity in row["entities"]:
                assert row["text"][entity["start"]:entity["end"]] == entity["text"]

print("All checks passed.")


In [ ]:
summary = pd.DataFrame(stats).T[["positive_examples", "negative_examples"]]
summary.plot(kind="bar")
plt.title("Positive and negative examples by split")
plt.ylabel("Number of examples")
plt.xticks(rotation=0)
plt.show()


## Leakage warning

Because templates and mountain names can appear across multiple splits, this
baseline evaluates sentence-level generalization. A stronger future experiment
should create an entity-disjoint test set with unseen mountain names.
